# 03 — Train HyperSpeech TokenMixer (scaffold)
This trains and caches HyperSpeech-TokenMixer (window-level) per outer fold.

In [ ]:
import os, json
import numpy as np
import torch
import yaml
from sklearn.pipeline import Pipeline

from src.data import DataSpec, load_dataframe, build_feature_sets
from src.splits import load_splits
from src.metrics import compute_binary_metrics, to_dict
from src.thresholding import choose_thresholds
from src.calibration import Calibrator
from src.models.wrappers_sklearn import build_preprocessor
from src.models.wrappers_torch import get_device, TorchTrainConfig, train_torch_binary, predict_proba_torch
from src.models.hyperspeech_tokenmixer import HyperSpeechTokenMixer, HyperSpeechTokenMixerConfig

cfg = yaml.safe_load(open("configs/base.yaml"))
hcfg = yaml.safe_load(open("configs/hyperspeech.yaml"))["hyperspeech"]

spec = DataSpec(
    csv_path=cfg["data"]["csv_path"],
    id_col=cfg["data"]["id_col"],
    target_col=cfg["data"]["target_col"],
    sbp_2classes_col=cfg["data"]["sbp_2classes_col"],
    positive_class_value=cfg["data"]["positive_class_value"],
    sbp_col=cfg["data"]["sbp_col"],
    sbp_threshold=cfg["data"]["sbp_threshold"],
    leakage_cols=tuple(cfg["data"]["leakage_cols"]),
    demographic_cols=tuple(cfg["data"]["demographic_cols"]),
    categorical_cols=tuple(cfg["data"]["categorical_cols"]),
)

df = load_dataframe(spec.csv_path)
splits = load_splits("artifacts/splits/outer_splits.json")
os.makedirs("artifacts/preds", exist_ok=True)
os.makedirs("artifacts/metrics", exist_ok=True)
os.makedirs("artifacts/models", exist_ok=True)

device = get_device(tuple(cfg["project"]["device_preference"]))
print("Device:", device)

class DenseTransformer:
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.toarray() if hasattr(X, "toarray") else X

for fs_name, fs_cfg in cfg["data"]["feature_sets"].items():
    X, y, groups, info, cat_cols = build_feature_sets(df, spec, include_demographics=fs_cfg["include_demographics"])

    pre = build_preprocessor(X, cat_cols=cat_cols)
    pre_pipe = Pipeline([("pre", pre), ("dense", DenseTransformer())])
    X_all = pre_pipe.fit_transform(X).astype(np.float32)
    y_all = y.values.astype(int)
    n_features = X_all.shape[1]

    token_mode = hcfg["tokenization"]["mode"]
    groups_list = None
    if token_mode == "group":
        n_groups = int(hcfg["tokenization"]["n_groups"])
        idxs = np.arange(n_features)
        groups_list = [g.tolist() for g in np.array_split(idxs, n_groups)]

    for fold, sp in enumerate(splits):
        tr_idx, te_idx = np.array(sp["train_idx"]), np.array(sp["test_idx"])
        tr_X, tr_y = X_all[tr_idx], y_all[tr_idx]
        te_X, te_y = X_all[te_idx], y_all[te_idx]

        n_val = max(1, int(0.1 * len(tr_X)))
        fit_X, fit_y = tr_X[:-n_val], tr_y[:-n_val]
        val_X, val_y = tr_X[-n_val:], tr_y[-n_val:]

        base_model = HyperSpeechTokenMixer(
            HyperSpeechTokenMixerConfig(
                n_features=n_features,
                d_token=int(hcfg["tokenmixer"]["d_token"]),
                n_layers=int(hcfg["tokenmixer"]["n_layers"]),
                dropout=float(hcfg["tokenmixer"]["dropout"]),
                gating=str(hcfg["tokenmixer"]["gating"]),
                feature_dropout=float(hcfg["tokenmixer"]["feature_dropout"]),
                token_mode=token_mode,
                groups=groups_list,
            )
        )

        class LogitOnly(torch.nn.Module):
            def __init__(self, m):
                super().__init__()
                self.m = m

            def forward(self, x):
                logits, _ = self.m(x)
                return logits

        model = LogitOnly(base_model)

        tcfg = TorchTrainConfig(
            lr=float(hcfg["optimization"]["lr"]),
            weight_decay=float(hcfg["optimization"]["weight_decay"]),
            batch_size=int(hcfg["optimization"]["batch_size"]),
            max_epochs=int(hcfg["optimization"]["max_epochs"]),
            patience=int(hcfg["optimization"]["early_stopping_patience"]),
            label_smoothing=float(hcfg["optimization"]["label_smoothing"]),
            device=str(device),
        )

        model = train_torch_binary(model, fit_X, fit_y, val_X, val_y, tcfg, device=device)

        p_tr = predict_proba_torch(model, tr_X, device=device)
        p_te = predict_proba_torch(model, te_X, device=device)

        if cfg["calibration"]["enabled"]:
            calib = Calibrator(method=cfg["calibration"]["method"]).fit(tr_y, p_tr)
            p_tr_c = calib.transform(p_tr)
            p_te_c = calib.transform(p_te)
        else:
            p_tr_c, p_te_c = p_tr, p_te

        thr = choose_thresholds(tr_y, p_tr_c, min_recall=cfg["screening"]["min_recall"])
        m_f1 = compute_binary_metrics(te_y, p_te_c, threshold=thr.threshold_f1)
        m_sc = compute_binary_metrics(te_y, p_te_c, threshold=thr.threshold_screening)

        json.dump(
            {
                "fs": fs_name,
                "model": "hyperspeech_tokenmixer",
                "fold": fold,
                "y_true": te_y.tolist(),
                "p_raw": p_te.tolist(),
                "p_cal": p_te_c.tolist(),
                "threshold_f1": thr.threshold_f1,
                "threshold_screen": thr.threshold_screening,
            },
            open(f"artifacts/preds/{fs_name}__hyperspeech_tokenmixer__fold{fold}.json", "w"),
            indent=2,
        )

        json.dump(
            {
                "fs": fs_name,
                "model": "hyperspeech_tokenmixer",
                "fold": fold,
                "f1_opt": to_dict(m_f1),
                "screening": to_dict(m_sc),
                "thresholds": {"f1": thr.threshold_f1, "screening": thr.threshold_screening},
            },
            open(f"artifacts/metrics/{fs_name}__hyperspeech_tokenmixer__fold{fold}.json", "w"),
            indent=2,
        )

        torch.save(model.state_dict(), f"artifacts/models/{fs_name}__hyperspeech_tokenmixer__fold{fold}.pt")
        print("[OK]", fs_name, "hyperspeech_tokenmixer fold", fold)